# Før du går i gang
**Vigtigt:** Lav først din egen kopi af denne notebook.

Vælg:
**File → Save a copy in Drive**

Du skal herefter arbejde i kopien i dit eget Google Drive.

Du behøver ikke bruge GitHub eller Git.

# Upload af datafiler

Hvis du skal arbejde med en datafil, f.eks. en `.txt`-fil, kan du uploade den direkte til din **Colab-session**.

1. Klik på **Filer** i menuen til venstre.
2. Klik på **Upload**-ikonet (Upload til sessionslager)
3. Vælg datafilen på din computer.
4. Filen vil nu kunne ses under **Files** og kan bruges i Python.

In [ ]:
from pyproj import Transformer, CRS
from numpy import genfromtxt, savetxt,column_stack
import matplotlib.pyplot as plt

import os
import pyproj
from pyproj import Transformer

!pip install -q pyproj
!wget -q https://raw.githubusercontent.com/holgerlindholm/DTU12821_Site-investigations_F2026/main/dk_sdfe_gvr2016.tif

pyproj_dir = pyproj.datadir.get_data_dir()
os.replace("dk_sdfe_gvr2016.tif", f"{pyproj_dir}/dk_sdfe_gvr2016.tif")

print(pyproj.datadir.get_data_dir())

print("✅ Så er alt downloadet og klar til brug")

Vi skal konvertere vores målte data om til at være i geoide højde, for at vi kan se hvilken vej vandet vil løbe.


In [ ]:
filnavn = r"/content/DRONE26.CSV"       # UDFYLD MED JERES EGET FILNAVN

data = genfromtxt(filnavn, skip_header=1, delimiter=",", dtype="3f8", usecols=(2,1,3))

Brug transformationer grupperegningen som inspiration til at udfylde nedenstående. 

I grupperegningen har vi antaget at vandstandsniveau er det samme som geoidehøjden, så derfor har vi brugt en geoide som grid. Med den info, hvilket grid kan der så bruges?

In [ ]:
# udfyld nedenstående ukendte for at lave transformationen
ellipsoide = 'GRS80'                # OBS: skal være skrevet med stort!
fra_koordinat_system = 'utm'        # OBS: skal være skrevet med småt!
zone = '22'                         # OBS: der skal kun være tal!
grid = 'gvr2016'                    # OBS: skal være skrevet med småt!

"""
UDFYLD IKKE NEDENSTÅENDE
"""
## Ved brug af PIPELINE
# pipeline = (
#     f"+ellps={ellipsoide} +proj=pipeline "
#     f"+step +proj={fra_koordinat_system} +zone={zone} +inv "
#     f"+step +proj=vgridshift +grids=dk_sdfe_{grid}.tif "
#     f"+step +proj={fra_koordinat_system} +zone={zone}"
# )

# transform_object = Transformer.from_pipeline(pipeline)

## Ved brug af CRS Koder (Se https://epsg.io/ og https://www.klimadatastyrelsen.dk/kortlaegning/geodaesi/koordinatsystemer)
kilde = CRS.from_epsg(3182).to_3d()        # GR96 / UTM 22N + ellipsoidehøjde
maal = CRS.from_user_input("EPSG:3182+8267")  # GR96 / UTM 22N + GVR2016 højde

transform_object = Transformer.from_crs(kilde, maal, always_xy=True, allow_ballpark=False)

koordinater_med_geoide_højde = transform_object.transform(*data.T)

print("De fem første koordinater i filen transformeret:")
print(f"Easting\t\t(x):\t{' '.join(str(k) for k in koordinater_med_geoide_højde[0][0:5])} meter")
print(f"Northing\t(y):\t{' '.join(str(k) for k in koordinater_med_geoide_højde[1][0:5])} meter")
print(f"Højder\t\t(z):\t{' '.join(str(k) for k in koordinater_med_geoide_højde[2][0:5])} meter")

Til sidst vil vi gerne gemme de transformerede data i en fil lokalt på jeres PC til brug i QGIS. Husk at det er den nye fil i gerne vil bruge til at undersøge hvilken vej vandet vil følge!

In [ ]:
basisnavn = os.path.splitext(filnavn)[0]

data_ud = column_stack(koordinater_med_geoide_højde)

savetxt(
    f"{basisnavn}_geoide.csv",
    data_ud,
    delimiter=",",
    fmt="%.4f",                              # undgår videnskabelig notation
    header="easting,northing,geoide_height",
    comments=""
)

print(f"Gemt som {basisnavn}_geoide.csv")